# Nevada conventional demonstration: 150 °C at 3 km

This notebook uses only public geoPFA interfaces and an explicit configuration cell. It starts from the processed JSON/layer tree produced by the standard Nevada workflow; local data and generated outputs are not version-controlled. The fitted GBLK model predicts the published Great Basin proxy-positive endpoint; a separate Stanford thermal-model panel reports `P(T(3 km) > 150 °C)`. Keeping these quantities separate avoids treating proxy labels as measured temperature outcomes.

## Load processed inputs and declare the model


In [ ]:
import copy
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import norm

from geopfa.layer_combination import VoterVeto
from geopfa.prob import ProbabilisticConfig, run_gblk_calibration_cv, run_probabilistic
from geopfa.prob import load_processed_pfa

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("geoPFA repository root not found")


def required_local_path(env_name: str, default: Path) -> Path:
    path = Path(os.environ.get(env_name, default)).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"Required local input is absent: {path}. Set {env_name} or prepare "
            "the documented public study data before running this notebook."
        )
    return path


def surface_on_raster_support(
    surface, *, raster_crs, raster_bounds, raster_resolution, name: str
):
    if surface.crs is None:
        raise ValueError(f"{name} surface must declare a CRS")
    if raster_crs is None or not raster_crs.is_projected:
        raise ValueError("thermal raster CRS must be projected")
    aligned = surface.to_crs(raster_crs)
    bounds = aligned.total_bounds
    if not np.isfinite(bounds).all():
        raise ValueError(f"{name} surface bounds must be finite")
    tolerance = max(abs(float(value)) for value in raster_resolution)
    outside = (
        bounds[0] < raster_bounds.left - tolerance
        or bounds[1] < raster_bounds.bottom - tolerance
        or bounds[2] > raster_bounds.right + tolerance
        or bounds[3] > raster_bounds.top + tolerance
    )
    if outside:
        raise ValueError(f"{name} surface falls outside the thermal raster support")
    return aligned

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / "examples" / "Nevada" / "2D"
output_root = Path(
    os.environ.get("GEOPFA_DEMO_OUTPUT_ROOT", project_dir / "outputs")
).expanduser().resolve()
output_dir = output_root / "nevada_conventional_150c_3km"
processed_config_path = required_local_path(
    "GEOPFA_NEVADA_PROCESSED_CONFIG", project_dir / "config" / "nevada_processed_config.json"
)
processed_data_dir = required_local_path(
    "GEOPFA_NEVADA_PROCESSED_DATA", project_dir / "data"
)
labels_path = required_local_path(
    "GEOPFA_NEVADA_LABELS",
    repo_root / "data" / "raw" / "nevada" / "labeled_wells_nevada.gpkg",
)
thermal_mean_path = required_local_path(
    "GEOPFA_NEVADA_3KM_MEAN",
    project_dir / "data" / "thermal" / "stanford_temperature_3km_mean_c.tif",
)
thermal_sd_path = required_local_path(
    "GEOPFA_NEVADA_3KM_SD",
    project_dir / "data" / "thermal" / "stanford_temperature_3km_sd_c.tif",
)
pfa, input_artifacts = load_processed_pfa(
    processed_config_path,
    processed_data_dir,
    crs="EPSG:26911",
)

config_dict = {'enabled': True,
 'output_dir': '../outputs/conventional_150c_3km',
 'dimensions': '2d',
 'labels': {'source': '../../../../data/labeled_wells/nevada_great_basin/labeled_wells_nevada.gpkg',
            'id_col': 'well_id',
            'label_columns': {'heat': 'heat_label'},
            'pu_mode': 'off',
            'min_wells_for_fit': 5},
 'alpha': {'heat': {'mode': 'scalar',
                    'scalar_fallback_pr0': 0.57,
                    'force_prior_predictive': False,
                    'use_evidence_prior': False}},
 'evidence': {'regularization': {'C': 0.1}},
 'spatial_field': {'enabled': True,
                   'backend': 'latticekrigx',
                   'n_levels': 2,
                   'lattice_centers_per_dimension': 3,
                   'coordinate_scaling': 'axis_range'},
 'inference': {'backend': 'gblk', 'gblk_bayesian': {'enabled': False}},
 'calibration': {'method': 'none', 'fit_on': 'block_cv'},
 'cross_validation': {'n_folds': 5,
                      'block_type': 'grid',
                      'block_size_km': 20.0,
                      'buffer_km': 10.0},
 'combination': {'rule': 'product'},
 'scenarios': [],
 'outputs': {'probability_rasters': True,
             'uncertainty_rasters': False,
             'format': ['geotiff', 'csv']}}
config_dict["output_dir"] = str(output_dir)
config_dict["labels"]["source"] = str(labels_path)
config = ProbabilisticConfig.from_dict(config_dict)
config

## Run both workflows and physical thermal context


In [ ]:
pfa_vv = VoterVeto.do_voter_veto(
    copy.deepcopy(pfa),
    normalize_method="minmax",
    component_veto=False,
    criteria_veto=True,
    normalize=True,
    norm_to=5,
)
input_artifacts.update({"thermal_mean": thermal_mean_path, "thermal_sd": thermal_sd_path})
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts=input_artifacts,
)
probability = model_result.combined.copy()
with rasterio.open(thermal_mean_path) as source:
    temperature_mean = source.read(1, masked=True).astype(float).filled(np.nan)
    thermal_crs = source.crs
    thermal_bounds = source.bounds
    thermal_resolution = source.res
    thermal_transform = source.transform
    thermal_extent = [source.bounds.left, source.bounds.right, source.bounds.bottom, source.bounds.top]
with rasterio.open(thermal_sd_path) as source:
    if (
        source.crs != thermal_crs
        or source.transform != thermal_transform
        or source.shape != temperature_mean.shape
    ):
        raise ValueError("thermal mean and uncertainty rasters must share one grid")
    temperature_sd = source.read(1, masked=True).astype(float).filled(np.nan)
valid_thermal = np.isfinite(temperature_mean) & np.isfinite(temperature_sd) & (temperature_sd > 0)
if not valid_thermal.any():
    raise ValueError("thermal rasters contain no finite mean with positive uncertainty")
thermal_probability = np.full(temperature_mean.shape, np.nan, dtype=float)
thermal_probability[valid_thermal] = norm.sf(
    (150.0 - temperature_mean[valid_thermal]) / temperature_sd[valid_thermal]
)
probability[["probability"]].describe()

## Compare the final surfaces


In [ ]:
vv = pfa_vv["pr_norm"].copy()
vv_plot = surface_on_raster_support(
    vv,
    raster_crs=thermal_crs,
    raster_bounds=thermal_bounds,
    raster_resolution=thermal_resolution,
    name="VoterVeto",
)
probability_plot = surface_on_raster_support(
    probability,
    raster_crs=thermal_crs,
    raster_bounds=thermal_bounds,
    raster_resolution=thermal_resolution,
    name="probability",
)
if len(vv_plot) != len(probability_plot):
    raise ValueError("VoterVeto and GBLK surfaces must share one ordered grid")
vv_geometry = vv_plot.geometry.reset_index(drop=True)
probability_geometry = probability_plot.geometry.reset_index(drop=True)
same_geometry = vv_geometry.geom_equals_exact(
    probability_geometry, tolerance=1e-6, align=False
)
if not bool(same_geometry.all()):
    raise ValueError("VoterVeto and GBLK surface geometries are not aligned")
comparison_mask = (
    np.isfinite(vv_plot["favorability"].to_numpy(dtype=float))
    & np.isfinite(probability_plot["probability"].to_numpy(dtype=float))
)
comparison_support_n = int(comparison_mask.sum())
if comparison_support_n == 0:
    raise ValueError("VoterVeto and GBLK surfaces have no shared finite support")
vv_comparison = vv_plot.iloc[np.flatnonzero(comparison_mask)].copy()
probability_comparison = probability_plot.iloc[np.flatnonzero(comparison_mask)].copy()
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), constrained_layout=True)
vv_comparison.plot(column="favorability", cmap="viridis", vmin=0, vmax=5, markersize=2, legend=True, ax=axes[0])
probability_comparison.plot(column="probability", cmap="magma", vmin=0, vmax=1, markersize=2, legend=True, ax=axes[1])
thermal_image = axes[2].imshow(thermal_probability, extent=thermal_extent, origin="upper", cmap="magma", vmin=0, vmax=1)
fig.colorbar(thermal_image, ax=axes[2], label="Probability")
axes[0].set_title("Traditional VoterVeto final score")
axes[1].set_title("GBLK P(proxy positive)")
axes[2].set_title("Stanford P(T at 3 km > 150 °C)")
for axis in axes:
    axis.set_xlim(thermal_bounds.left, thermal_bounds.right)
    axis.set_ylim(thermal_bounds.bottom, thermal_bounds.top)
    axis.set_axis_off()
fig.suptitle(f"VoterVeto and GBLK shared finite support: n={comparison_support_n:,}")
plt.show()

## Evaluate held-out spatial folds


In [ ]:
cv = run_gblk_calibration_cv(
    pfa,
    config,
    components=("heat",),
    n_folds=5,
    random_state=0,
)["heat"]

def fold_ece(fold) -> float:
    reliability = fold.reliability
    populated = reliability.bin_counts > 0
    counts = reliability.bin_counts[populated]
    gaps = np.abs(
        reliability.bin_mean_pred[populated] - reliability.bin_fracs[populated]
    )
    return float(np.sum(counts * gaps) / np.sum(counts))

metric_distributions = {
    "brier_score": [fold.brier_score for fold in cv.folds],
    "brier_skill_score": [fold.brier_skill_score for fold in cv.folds],
    "expected_calibration_error": [fold_ece(fold) for fold in cv.folds],
}
fold_metrics = pd.DataFrame(
    {
        "fold": [fold.fold_id for fold in cv.folds],
        "n_train": [fold.extra["n_train"] for fold in cv.folds],
        "n_test": [fold.n_test for fold in cv.folds],
        "n_buffered": [fold.extra["n_buffered"] for fold in cv.folds],
        **metric_distributions,
    }
)
fold_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
axes[0].bar(fold_metrics.fold - 0.18, fold_metrics.brier_score, width=0.36, label="Brier score")
axes[0].bar(fold_metrics.fold + 0.18, fold_metrics.brier_skill_score, width=0.36, label="Brier skill")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(xlabel="Spatial fold", ylabel="Score", title="Held-out spatial-fold Brier metrics")
axes[0].legend()
axes[1].bar(fold_metrics.fold, fold_metrics.expected_calibration_error, color="#6A3D9A")
axes[1].set(xlabel="Spatial fold", ylabel="ECE", title="Held-out spatial-fold calibration error")
plt.show()
{
    "fold_metric_distribution": fold_metrics.describe().to_dict(),
    "comparison_support_n": comparison_support_n,
    "thermal_probability_mean": float(np.nanmean(thermal_probability)),
    "thermal_probability_max": float(np.nanmax(thermal_probability)),
}

## Interpretation and caveat

The five held-out units are spatial folds, not independent replicate datasets. Brier score and Brier skill assess out-of-fold probability quality, while ECE summarizes calibration; these metrics do not by themselves establish discrimination. They apply to the 145-site GDR1351 proxy design (83 positives and 62 designed pseudo-absences), not to direct temperature measurements or discovery of an economic resource. The VoterVeto and GBLK panels are restricted to their shared finite support, whose cell count is reported above, but their values retain different meanings and scales. The 3 km thermal probability is independently defined physical context from a regional thermal model.